# RecBinarySearch and ItBinarySearch (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/RecBinarySearch_ItBinarySearch_java.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `RecBinarySearch` and `ItBinarySearch`: the algorithms in Java, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/RecBinarySearch_ItBinarySearch_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/RecBinarySearch_ItBinarySearch_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fibRec` (camelCase).

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Tail recursion: RecBinarySearch and ItBinarySearch

A recursive call is a *tail call* if it is the last operation and its result is returned as it is: it can be replaced by an update of the parameters and a jump to the beginning, that is, by a loop (loop condition = negation of the base case; tail call = simultaneous assignment of the parameters). The result is the same, the number of steps has the same order of growth and the extra memory goes down from $\Theta(\text{depth})$ to $\Theta(1)$.

**RecBS**$(a,l,r,m)$ / **ItBS**$(a,l,r,m)$ — *Input*: a sorted array, two indices $l$, $r$ and a value $m$. *Output*: an index of $m$ in $a[l..r]$ or $-1$. `ItBS(a, 1, n, m)` is the iterative binary search of the chapter "Search". In the pseudo-code of the chapter they are called `RecBinarySearch` and `ItBinarySearch`.

On the array $a=[3,9,11,12,12,16,19]$ of the chapter we search for $m=12,10,3,19,100$: for each value the returned index (from 0), the calls of `RecBS` and the iterations of `ItBS`.

In [2]:
%%writefile RecItBinarySearch.java
import java.util.*;

public class RecItBinarySearch {
    static int rbsCalls = 0;   // calls of RecBS
    static int bsIters = 0;    // iterations of ItBS

    // recBS(a, l, r, m)
    static int recBS(int[] a, int l, int r, int m) {
        rbsCalls++;
        if (l > r) {
            return -1;
        }
        int mid = l + (r - l) / 2;
        if (a[mid] == m) {
            return mid;
        }
        if (a[mid] < m) {
            return recBS(a, mid + 1, r, m);     // tail call
        } else {
            return recBS(a, l, mid - 1, m);     // tail call
        }
    }

    // itBS(a, l, r, m)
    static int itBS(int[] a, int l, int r, int m) {
        while (l <= r) {                        // negation of the base case l > r
            bsIters++;
            int mid = l + (r - l) / 2;
            if (a[mid] == m) {
                return mid;                     // the second base case stays in the loop
            }
            if (a[mid] < m) {
                l = mid + 1;
            } else {
                r = mid - 1;
            }
        }
        return -1;
    }

    public static void main(String[] args) {
        int[] a = {3, 9, 11, 12, 12, 16, 19};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.printf("%4s %6s %6s %6s %11s%n", "m", "RecBS", "calls", "ItBS", "iterations");
        int[] ms = {12, 10, 3, 19, 100};
        for (int m : ms) {
            rbsCalls = 0;
            int r1 = recBS(a, 0, n - 1, m);
            bsIters = 0;
            int r2 = itBS(a, 0, n - 1, m);
            System.out.printf("%4d %6d %6d %6d %11d   %s%n", m, r1, rbsCalls, r2, bsIters, r1 == r2 ? "OK" : "ERROR");
        }
    }
}

Writing RecItBinarySearch.java


In [3]:
!javac RecItBinarySearch.java && java RecItBinarySearch

a = [3, 9, 11, 12, 12, 16, 19]   n = 7
   m  RecBS  calls   ItBS  iterations
  12      3      1      3           1   OK
  10     -1      4     -1           3   OK
   3      0      3      0           3   OK
  19      6      3      6           3   OK
 100     -1      4     -1           3   OK


**Try it.** Search for other values, present or absent, in another sorted array: compare the calls of `RecBS` with the iterations of `ItBS`.

In [4]:
%%writefile ProvaRecItBinarySearch.java
import java.util.*;

public class ProvaRecItBinarySearch {
    public static void main(String[] args) {
        // Search for other values, present or absent: compare the calls of RecBS with the iterations of ItBS.
        int[] b = {2, 5, 8, 11, 14, 17, 20, 23};
        int n = b.length;
        int[] ms = {14, 1, 30};
        for (int m : ms) {
            RecItBinarySearch.rbsCalls = 0;
            int r1 = RecItBinarySearch.recBS(b, 0, n - 1, m);
            RecItBinarySearch.bsIters = 0;
            int r2 = RecItBinarySearch.itBS(b, 0, n - 1, m);
            System.out.println(m + " " + r1 + " " + RecItBinarySearch.rbsCalls + " " + r2 + " " + RecItBinarySearch.bsIters);
        }
    }
}

Writing ProvaRecItBinarySearch.java


In [5]:
!javac ProvaRecItBinarySearch.java && java ProvaRecItBinarySearch

14 4 3 4 3
1 -1 4 -1 3
30 -1 5 -1 4


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).